<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/evaluacion_ragas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB2 · Evaluación RAGAS (Fase 1)

Evalúa las 171 respuestas generadas por NB1 (`respuestas_gpt4o.json`).

**Entorno AISLADO de NB1**: RAGAS 0.4 exige langchain-core 0.3.x, incompatible con el 1.x de NB1.

**Métricas**: Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference + la custom `deteccion_de_vigencia` (el corazón de la tesis).

**Orden**: correr celda por celda. La celda 5 tiene `MODO_PRUEBA=True` (6 respuestas). Verificar que sale bien y recién ahí `MODO_PRUEBA=False` para las 171.

## 1. Dependencias (pines exactos — no cambiar)

In [ ]:
# ============================================================================
# NB2 · evaluacion_ragas.ipynb — CELDA 1: dependencias (entorno AISLADO)
# ============================================================================
# ⚠️ ESTE NOTEBOOK CORRE EN UN ENTORNO SEPARADO DE NB1.
#    RAGAS 0.4.3 exige langchain-core 0.3.x, que es INCOMPATIBLE con el
#    langchain-core 1.x que usa NB1 (LangGraph). Por eso son notebooks distintos:
#    NB1 genera los JSON, NB2 los evalúa. No se ejecutan juntos.
#
# ⚠️ pip install ragas "a secas" está ROTO (bug upstream: importa un módulo de
#    langchain-community que la 0.4 eliminó). Hay que PINEAR estas versiones exactas.
# ----------------------------------------------------------------------------
!pip install -q \
  "ragas==0.4.3" \
  "langchain-core==0.3.86" \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-openai==0.3.35" \
  "openai>=1.0.0"

# verificación de que importa (si esto falla, revisar los pines de arriba)
import ragas
print("ragas", ragas.__version__, "— import OK")


## 2. Setup: Drive, API key, cargar los JSON de NB1

In [ ]:
# ============================================================================
# CELDA 2: setup — Drive, API key, cargar los JSON de NB1
# ============================================================================
import os, json, getpass
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# API key de OpenAI (el juez de RAGAS es GPT-4o)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

# --- rutas (misma estructura que NB1) ---
CARPETA = "/content/drive/MyDrive/tesis_chatbot/ragas"
PATH_RESP = os.path.join(CARPETA, "respuestas_gpt4o.json")

with open(PATH_RESP, encoding="utf-8") as f:
    data = json.load(f)

META = data["meta"]
RESP = data["respuestas"]
print("Cargado:", PATH_RESP)
print("Meta:", META)
print("Total respuestas:", len(RESP), "(esperado 171 = 57×3)")

# índice rápido por (id, brazo)
POR_CLAVE = {(r["id"], r["brazo"]): r for r in RESP}
BRAZOS = sorted({r["brazo"] for r in RESP})
IDS = sorted({r["id"] for r in RESP})
print("Brazos:", BRAZOS)
print("Preguntas:", len(IDS))


## 3. Juez (GPT-4o) y las 4 métricas estándar

In [ ]:
# ============================================================================
# CELDA 3: instanciar el JUEZ (LLM + embeddings) y las MÉTRICAS de RAGAS
# ============================================================================
# El juez es GPT-4o (mismo modelo que generó, pero como los 3 brazos usan el
# mismo generador, el sesgo de auto-preferencia es SIMÉTRICO y se cancela en la
# comparación entre brazos. Documentarlo en Limitaciones).
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings

_client = OpenAI()   # usa OPENAI_API_KEY del entorno

# LLM juez y embeddings (para AnswerRelevancy)
judge_llm = llm_factory("gpt-4o", client=_client)
judge_emb = RagasEmbeddings(client=_client, model="text-embedding-3-small")

# --- Las 4 métricas estándar (API collections, RAGAS 0.4) ---
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference
)
m_faith   = Faithfulness(llm=judge_llm)
m_relev   = AnswerRelevancy(llm=judge_llm, embeddings=judge_emb)
m_recall  = ContextRecall(llm=judge_llm)                 # necesita reference (gold)
m_precis  = ContextPrecisionWithReference(llm=judge_llm)  # necesita reference (gold)

print("Juez y métricas estándar listos.")
print("Firmas (recordatorio):")
print("  Faithfulness.ascore(user_input, response, retrieved_contexts)   -> sin gold")
print("  AnswerRelevancy.ascore(user_input, response)                    -> sin gold")
print("  ContextRecall.ascore(user_input, retrieved_contexts, reference) -> CON gold")
print("  ContextPrecisionWithReference.ascore(user_input, reference, retrieved_contexts) -> CON gold")


## 4. Métrica custom: detección de vigencia

La única que detecta el fallo central del baseline (citar ley derogada como vigente). Usa el campo `rubrica` del set gold como criterio.

In [ ]:
# ============================================================================
# CELDA 4: MÉTRICA CUSTOM · deteccion_de_vigencia  (el corazón de la tesis)
# ============================================================================
# Ninguna de las 4 métricas estándar detecta el fallo central del baseline:
# citar una ley DEROGADA como si estuviera vigente (Faithfulness incluso premia
# eso si el baseline resume fielmente un contexto viejo). Esta métrica end-to-end
# pregunta directamente: ¿la respuesta enuncia bien el estado de vigencia?
#
# ⚠️ AspectCritic fue ELIMINADO en RAGAS 0.4. Se usa el decorador @discrete_metric.
# La rúbrica de cada pregunta (campo 'rubrica' del set gold) es el CRITERIO que
# se le pasa al juez.
from ragas.metrics import discrete_metric
from ragas.metrics.result import MetricResult
from openai import OpenAI

_c = OpenAI()

PROMPT_VIGENCIA = """Sos un evaluador jurídico. Determiná si la RESPUESTA enuncia correctamente
el estado de vigencia de las normas involucradas, según el CRITERIO dado.

CRITERIO (qué debe cumplir una respuesta correcta):
{rubrica}

HECHOS DE REFERENCIA (verdad sobre la vigencia):
{gold}

RESPUESTA A EVALUAR:
{response}

Respondé con UNA sola palabra:
- "correcto"   si la respuesta respeta el criterio y no presenta como vigente una norma derogada.
- "incorrecto" si presenta una norma derogada como vigente, o contradice el criterio de vigencia.
- "no_aplica"  si la pregunta no involucra vigencia de normas.
Palabra:"""

@discrete_metric(name="deteccion_de_vigencia",
                 allowed_values=["correcto", "incorrecto", "no_aplica"])
def deteccion_de_vigencia(response: str, gold: str, rubrica: str) -> MetricResult:
    # sin criterio ni gold no se puede juzgar vigencia
    if not (rubrica or gold):
        return MetricResult(value="no_aplica", reason="sin rúbrica ni gold de vigencia")
    prompt = PROMPT_VIGENCIA.format(rubrica=rubrica or "(no especificado)",
                                    gold=gold or "(no especificado)",
                                    response=response)
    out = _c.chat.completions.create(
        model="gpt-4o", temperature=0,
        messages=[{"role": "user", "content": prompt}],
    ).choices[0].message.content.strip().lower()
    val = "correcto" if "correcto" in out else ("incorrecto" if "incorrecto" in out else "no_aplica")
    return MetricResult(value=val, reason=out[:120])

# prueba rápida en una respuesta del baseline (debería dar 'incorrecto' en una de reforma)
_demo = POR_CLAVE.get(("P1-01", "A_baseline"))
if _demo:
    r = deteccion_de_vigencia.score(response=_demo["respuesta"],
                                    gold=_demo["gold"], rubrica=_demo["rubrica"])
    print("Demo P1-01 / A_baseline →", r.value, "|", r.reason[:80])


## 5. Ejecutar la evaluación

⚠️ Empezá con `MODO_PRUEBA=True`. Si las 6 salen sin error, pasá a `False` (171). Throttling incluido por el límite de 30k tokens/min de la org.

In [ ]:
# ============================================================================
# CELDA 5: EJECUTAR la evaluación sobre las 171 respuestas
# ============================================================================
# Aplica: las 4 métricas estándar + deteccion_de_vigencia.
# Reglas especiales (del ESTADO §16.5):
#   - fuera_alcance: se EXCLUYE de AnswerRelevancy (castiga el fallback honesto con 0).
#     En su lugar se marca 'fallback_ok' (¿admitió no saber?).
#   - reference (gold) vacío: se saltan ContextRecall y ContextPrecisionWithReference.
# Throttling: la org tiene TPM=30k. Se corre con pausa y reintentos para los 429.
import asyncio, time, json, math
from datetime import datetime

MODO_PRUEBA = True      # <-- True: solo 6 respuestas (2 preguntas × 3 brazos). False: las 171.
N_PRUEBA = 6
PAUSA = 1.5             # segundos entre preguntas (respeta el TPM)

def es_fuera_alcance(r):  return "fuera_alcance" in (r.get("tipo") or [])
def texto_pregunta(r):    return r.get("pregunta_usada") or r.get("pregunta_original")

async def evaluar_una(r):
    q = texto_pregunta(r)
    resp = r["respuesta"]
    ctxs = r["contextos"]
    gold = r.get("gold") or ""
    rub  = r.get("rubrica") or ""
    fila = {"id": r["id"], "brazo": r["brazo"],
            "afectada_por_reforma": r.get("afectada_por_reforma", False),
            "tipo": r.get("tipo") or []}

    # --- Faithfulness (siempre) ---
    try:
        fila["faithfulness"] = (await m_faith.ascore(
            user_input=q, response=resp, retrieved_contexts=ctxs)).value
    except Exception as e:
        fila["faithfulness"] = None; fila["err_faith"] = str(e)[:80]

    # --- AnswerRelevancy (EXCLUYE fuera_alcance) ---
    if es_fuera_alcance(r):
        fila["answer_relevancy"] = None
        fila["fallback_ok"] = any(k in resp.lower() for k in
            ["no tengo", "no cuento", "no dispongo", "no puedo", "fuera de", "no encontré", "no hay información"])
    else:
        try:
            fila["answer_relevancy"] = (await m_relev.ascore(
                user_input=q, response=resp)).value
        except Exception as e:
            fila["answer_relevancy"] = None; fila["err_relev"] = str(e)[:80]

    # --- ContextRecall + Precision (solo si hay gold) ---
    if gold:
        try:
            fila["context_recall"] = (await m_recall.ascore(
                user_input=q, retrieved_contexts=ctxs, reference=gold)).value
        except Exception as e:
            fila["context_recall"] = None; fila["err_recall"] = str(e)[:80]
        try:
            fila["context_precision"] = (await m_precis.ascore(
                user_input=q, reference=gold, retrieved_contexts=ctxs)).value
        except Exception as e:
            fila["context_precision"] = None; fila["err_precis"] = str(e)[:80]
    else:
        fila["context_recall"] = None
        fila["context_precision"] = None

    # --- deteccion_de_vigencia (métrica custom, síncrona) ---
    try:
        rv = deteccion_de_vigencia.score(response=resp, gold=gold, rubrica=rub)
        fila["deteccion_vigencia"] = rv.value
    except Exception as e:
        fila["deteccion_vigencia"] = None; fila["err_vig"] = str(e)[:80]

    return fila

async def correr(lista):
    out = []
    for i, r in enumerate(lista, 1):
        for intento in range(4):     # reintentos ante 429
            try:
                out.append(await evaluar_una(r)); break
            except Exception as e:
                if "rate" in str(e).lower() and intento < 3:
                    espera = 2 ** intento
                    print(f"  429, esperando {espera}s..."); time.sleep(espera)
                else:
                    out.append({"id": r["id"], "brazo": r["brazo"], "error": str(e)[:100]}); break
        print(f"[{i}/{len(lista)}] {r['id']}/{r['brazo']}")
        time.sleep(PAUSA)
    return out

lote = RESP[:N_PRUEBA] if MODO_PRUEBA else RESP
print(f"{'PRUEBA' if MODO_PRUEBA else 'COMPLETO'}: evaluando {len(lote)} respuestas...")
resultados = await correr(lote)

# guardar
sufijo = "_PRUEBA" if MODO_PRUEBA else ""
out_path = os.path.join(CARPETA, f"metricas_ragas{sufijo}.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({"meta": {"generado": datetime.now().isoformat(), "n": len(resultados)},
               "resultados": resultados}, f, ensure_ascii=False, indent=2)
print("\n✓ Guardado:", out_path)


## 6. Análisis — la tabla de la tesis

Promedios por brazo + el titular: detección de vigencia en las 17 preguntas de la reforma.

In [ ]:
# ============================================================================
# CELDA 6: ANÁLISIS — la tabla que va a la tesis
# ============================================================================
# Promedios por brazo de cada métrica, más el desglose de deteccion_de_vigencia
# sobre las preguntas afectadas por la reforma (el titular de la tesis).
import pandas as pd

df = pd.DataFrame(resultados)
if "error" in df.columns:
    err = df[df["error"].notna()]
    if len(err): print("⚠ Filas con error:", len(err)); display(err[["id","brazo","error"]])

MET_NUM = ["faithfulness","answer_relevancy","context_recall","context_precision"]

# --- 1) Promedios por brazo (métricas numéricas) ---
print("="*60); print("PROMEDIOS POR BRAZO (métricas RAGAS estándar)"); print("="*60)
tabla = df.groupby("brazo")[MET_NUM].mean(numeric_only=True).round(3)
display(tabla)

# --- 2) deteccion_de_vigencia: conteo por brazo ---
print("\n"+"="*60); print("DETECCIÓN DE VIGENCIA — conteo por brazo"); print("="*60)
piv = df.pivot_table(index="brazo", columns="deteccion_vigencia",
                     values="id", aggfunc="count", fill_value=0)
display(piv)

# --- 3) EL TITULAR: vigencia SOLO en las afectadas por la reforma ---
print("\n"+"="*60); print("★ TITULAR: vigencia en las preguntas de la REFORMA"); print("="*60)
ref = df[df["afectada_por_reforma"] == True]
if len(ref):
    piv_ref = ref.pivot_table(index="brazo", columns="deteccion_vigencia",
                              values="id", aggfunc="count", fill_value=0)
    display(piv_ref)
    # tasa de acierto por brazo
    print("\nTasa de 'correcto' sobre afectadas por reforma:")
    for b in sorted(ref["brazo"].unique()):
        sub = ref[ref["brazo"] == b]
        ok = (sub["deteccion_vigencia"] == "correcto").sum()
        print(f"  {b}: {ok}/{len(sub)}")

# --- 4) fallback en fuera_alcance ---
if "fallback_ok" in df.columns:
    print("\n"+"="*60); print("FALLBACK HONESTO (fuera_alcance)"); print("="*60)
    fa = df[df["fallback_ok"].notna()]
    for b in sorted(fa["brazo"].unique()):
        sub = fa[fa["brazo"] == b]
        ok = sub["fallback_ok"].sum()
        print(f"  {b}: {ok}/{len(sub)} admitió no saber")

# --- 5) guardar la tabla resumen ---
tabla.to_csv(os.path.join(CARPETA, "resumen_metricas.csv"))
print("\n✓ Resumen guardado en resumen_metricas.csv")


## Notas

- El juez GPT-4o evalúa respuestas de GPT-4o: sesgo de auto-preferencia SIMÉTRICO entre brazos (se cancela en la comparación). Documentar en Limitaciones.
- `deteccion_de_vigencia` sobre las afectadas por reforma = el resultado central.
- Salidas en Drive: `metricas_ragas.json`, `resumen_metricas.csv`.